# SmartStudy AI — Gemma 4 E4B Fine-Tuning Pipeline

**3-Stage Pipeline:** SFT+rsLoRA → GRPO → SimPO → GGUF Export

Run this notebook on **Kaggle with T4 GPU** (free tier).

Built for the [Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon) — Education Track.

## 0. Setup & Dependencies

In [ ]:
%%capture
!pip install unsloth trl peft datasets accelerate bitsandbytes
# Unsloth handles the heavy lifting for efficient fine-tuning on T4

In [ ]:
import json
import random
import re
from pathlib import Path

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Dataset Preparation

Pull from SciQ, ARC, OpenBookQA, MMLU and format into chat-style JSONL.

In [ ]:
from datasets import load_dataset, Dataset

BLOOM_LEVELS = ["remember", "understand", "apply", "analyze", "evaluate", "create"]
DIFFICULTIES = ["easy", "medium", "hard"]
random.seed(42)


def format_mcq(question, options, correct, explanation, difficulty="medium", bloom="understand"):
    return {
        "question": question, "type": "mcq", "options": options[:4],
        "correct_answer": correct, "explanation": explanation,
        "difficulty": difficulty, "bloom_level": bloom,
    }


def build_chat_example(mcq, user_msg):
    return {
        "messages": [
            {"role": "user", "content": user_msg},
            {"role": "model", "content": json.dumps({"questions": [mcq]}, ensure_ascii=False)},
        ]
    }


all_examples = []

# --- SciQ (2000) ---
print("Loading SciQ...")
sciq = load_dataset("allenai/sciq", split="train")
for row in sciq:
    if len(all_examples) >= 2000:
        break
    correct = row["correct_answer"]
    options = [correct, row["distractor1"], row["distractor2"], row["distractor3"]]
    random.shuffle(options)
    explanation = row.get("support", "") or f"The correct answer is {correct}."
    mcq = format_mcq(row["question"], options, correct, explanation,
                     random.choice(DIFFICULTIES), random.choice(BLOOM_LEVELS[:3]))
    all_examples.append(build_chat_example(
        mcq, "Generate 1 MCQ question about science. Output valid JSON."
    ))
print(f"  SciQ: {len(all_examples)}")

# --- ARC (1500) ---
print("Loading ARC...")
arc_count = len(all_examples)
for split_name in ["ARC-Easy", "ARC-Challenge"]:
    difficulty = "easy" if "Easy" in split_name else "hard"
    ds = load_dataset("allenai/ai2_arc", split_name, split="train")
    for row in ds:
        if len(all_examples) - arc_count >= 1500:
            break
        labels, texts = row["choices"]["label"], row["choices"]["text"]
        try:
            correct = texts[labels.index(row["answerKey"])]
        except (ValueError, IndexError):
            continue
        mcq = format_mcq(row["question"], texts[:4], correct,
                         f"The correct answer is '{correct}'.",
                         difficulty, random.choice(BLOOM_LEVELS[:4]))
        all_examples.append(build_chat_example(
            mcq, f"Generate 1 {difficulty} science MCQ. Output valid JSON."
        ))
print(f"  ARC: {len(all_examples) - arc_count}")

# --- OpenBookQA (800) ---
print("Loading OpenBookQA...")
obqa_count = len(all_examples)
ds = load_dataset("allenai/openbookqa", "main", split="train")
for row in ds:
    if len(all_examples) - obqa_count >= 800:
        break
    labels, texts = row["choices"]["label"], row["choices"]["text"]
    try:
        correct = texts[labels.index(row["answerKey"])]
    except (ValueError, IndexError):
        continue
    fact = row.get("fact1", "")
    explanation = f"The correct answer is '{correct}'." + (f" Key fact: {fact}" if fact else "")
    mcq = format_mcq(row["question_stem"], texts[:4], correct, explanation,
                     random.choice(DIFFICULTIES), random.choice(BLOOM_LEVELS))
    all_examples.append(build_chat_example(mcq, "Generate 1 conceptual MCQ. Output valid JSON."))
print(f"  OpenBookQA: {len(all_examples) - obqa_count}")

# --- MMLU (700) ---
print("Loading MMLU...")
mmlu_count = len(all_examples)
target_subjects = [
    "high_school_biology", "high_school_chemistry", "high_school_physics",
    "high_school_world_history", "high_school_us_history",
    "high_school_geography", "high_school_psychology",
    "college_biology", "college_chemistry",
]
ds = load_dataset("cais/mmlu", "all", split="test")
for row in ds:
    if len(all_examples) - mmlu_count >= 700:
        break
    if row.get("subject") not in target_subjects:
        continue
    choices = row["choices"]
    idx = row["answer"]
    if len(choices) < 4 or idx >= len(choices):
        continue
    subject = row["subject"].replace("_", " ").title()
    mcq = format_mcq(row["question"], choices[:4], choices[idx],
                     f"The correct answer is '{choices[idx]}'.",
                     "hard", random.choice(BLOOM_LEVELS[2:]))
    all_examples.append(build_chat_example(mcq, f"Generate 1 advanced {subject} MCQ. Output valid JSON."))
print(f"  MMLU: {len(all_examples) - mmlu_count}")

random.shuffle(all_examples)
split_idx = int(len(all_examples) * 0.9)
train_data = all_examples[:split_idx]
eval_data = all_examples[split_idx:]
print(f"\nTotal: {len(all_examples)} | Train: {len(train_data)} | Eval: {len(eval_data)}")

## 2. Stage 1 — SFT + rsLoRA

Teach Gemma 4 E4B the output format and domain knowledge.

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "unsloth/gemma-4-E4B-it"
MAX_SEQ_LENGTH = 2048

print("Loading Gemma 4 E4B...")
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True, use_gradient_checkpointing="unsloth",
)

print("Attaching rsLoRA adapters (r=32, all linear layers)...")
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_rslora=True,
)

In [ ]:
# Format for chat template
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_ds = Dataset.from_list(train_data).map(format_chat)
eval_ds = Dataset.from_list(eval_data).map(format_chat)

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")
print(f"Sample:\n{train_ds[0]['text'][:500]}")

In [ ]:
training_args = SFTConfig(
    output_dir="./stage1_sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    bf16=True,
    max_seq_length=MAX_SEQ_LENGTH,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    tokenizer=tokenizer,
)

print("Starting Stage 1 SFT...")
stats = trainer.train()
print(f"Done! Steps: {stats.global_step} | Loss: {stats.training_loss:.4f}")

model.save_pretrained("./stage1_sft/final")
tokenizer.save_pretrained("./stage1_sft/final")
print("Stage 1 saved.")

## 3. Stage 2 — GRPO (Group Relative Policy Optimization)

Teach the model to *reason about quiz quality* using multi-dimensional reward functions.
Same method DeepSeek-R1 used for reasoning.

In [ ]:
# === Reward Functions (inlined for notebook) ===

def _try_parse_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r'```(?:json)?\s*\n?(.*?)\n?```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    return None


def _score_quiz(text):
    reward = 0.0
    parsed = _try_parse_json(text)
    if parsed is None:
        return 0.0

    reward += 0.15  # Valid JSON
    questions = parsed.get("questions", [])
    if isinstance(questions, list) and len(questions) > 0:
        reward += 0.15  # Correct schema
    if not questions:
        return reward

    # Schema completeness (0.15)
    required = {"question", "correct_answer", "options"}
    for q in questions:
        if isinstance(q, dict) and required.issubset(q.keys()):
            reward += 0.15 / len(questions)

    # Answer validity (0.05) — correct_answer must be in options
    for q in questions:
        if isinstance(q, dict):
            opts = q.get("options", [])
            correct = q.get("correct_answer", "")
            if isinstance(opts, list) and correct and correct in opts:
                reward += 0.05 / len(questions)

    # Explanation quality (0.2)
    explanation_score = 0.0
    for q in questions:
        if not isinstance(q, dict): continue
        expl = q.get("explanation", "")
        if len(expl) > 50: explanation_score += 0.5
        if len(expl) > 100: explanation_score += 0.25
        causal = ["because", "this is", "the reason", "for example", "since", "therefore", "due to"]
        if any(w in expl.lower() for w in causal): explanation_score += 0.25
    reward += 0.2 * min(1.0, explanation_score / len(questions))

    # Difficulty distribution (0.1)
    diffs = set(q.get("difficulty", "") for q in questions if isinstance(q, dict))
    valid_diffs = diffs & {"easy", "medium", "hard"}
    if len(valid_diffs) >= 2: reward += 0.05
    if len(valid_diffs) >= 3: reward += 0.05

    # Bloom's coverage (0.1)
    valid_blooms = {"remember", "understand", "apply", "analyze", "evaluate", "create"}
    blooms = set(q.get("bloom_level", "") for q in questions if isinstance(q, dict))
    if len(blooms & valid_blooms) >= 2: reward += 0.05
    if len(blooms & valid_blooms) >= 3: reward += 0.05

    # Option count (0.1)
    for q in questions:
        if isinstance(q, dict):
            opts = q.get("options", [])
            if isinstance(opts, list) and len(opts) == 4:
                reward += 0.1 / len(questions)

    # Repetition penalty
    q_texts = [q.get("question", "").lower().strip() for q in questions if isinstance(q, dict)]
    if len(q_texts) != len(set(q_texts)):
        reward -= 0.2

    return max(0.0, min(1.0, reward))


def quiz_reward_function(completions, prompts=None, **kwargs):
    return [_score_quiz(c if isinstance(c, str) else str(c)) for c in completions]


# Quick sanity check
good = json.dumps({"questions": [{
    "question": "What is X?", "type": "mcq",
    "options": ["A", "B", "C", "D"], "correct_answer": "A",
    "explanation": "A is correct because it represents the primary mechanism by which cells generate energy through oxidative phosphorylation.",
    "difficulty": "medium", "bloom_level": "understand",
}]})
print(f"Good quiz score: {_score_quiz(good):.2f}")
print(f"Bad quiz score:  {_score_quiz('not json'):.2f}")

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Reload Stage 1 model
print("Loading Stage 1 model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    "./stage1_sft/final", max_seq_length=2048, load_in_4bit=True,
)

# Create diverse GRPO prompts
subjects = ["Biology", "Chemistry", "Physics", "Mathematics", "History",
            "Geography", "Psychology", "Computer Science", "Literature", "Economics"]
difficulties = ["easy", "medium", "hard", "mixed"]
grade_levels = ["Grade 8", "Grade 10", "Grade 12", "Undergraduate"]
num_q_options = [3, 5, 7, 10]

grpo_prompts = []
for i in range(500):
    s = random.choice(subjects)
    d = random.choice(difficulties)
    g = random.choice(grade_levels)
    n = random.choice(num_q_options)
    grpo_prompts.append({"prompt": (
        f"Generate {n} {d} MCQ questions about {s} for {g} students. "
        f'Output valid JSON with the schema: '
        f'{{"questions": [{{"question", "type": "mcq", "options": [4], '
        f'"correct_answer", "explanation", "difficulty", "bloom_level"}}]}}'
    )})

grpo_dataset = Dataset.from_list(grpo_prompts)
print(f"GRPO prompts: {len(grpo_dataset)}")

In [ ]:
grpo_config = GRPOConfig(
    output_dir="./stage2_grpo",
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_completion_length=1024,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    kl_coef=0.05,
)

trainer = GRPOTrainer(
    model=model, args=grpo_config,
    train_dataset=grpo_dataset,
    reward_funcs=[quiz_reward_function],
    tokenizer=tokenizer,
)

print("Starting Stage 2 GRPO...")
stats = trainer.train()
print(f"Done! Steps: {stats.global_step}")

model.save_pretrained("./stage2_grpo/final")
tokenizer.save_pretrained("./stage2_grpo/final")
print("Stage 2 saved.")

## 4. Stage 3 — SimPO (Simple Preference Optimization)

Reference-free preference learning via self-play. Generates N responses, scores them,
keeps best/worst pairs with quality gap > 0.2.

In [ ]:
from trl import SimPOConfig, SimPOTrainer

# Reload Stage 2 model
print("Loading Stage 2 model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    "./stage2_grpo/final", max_seq_length=2048, load_in_4bit=True,
)

# Generate preference pairs via self-play
FastLanguageModel.for_inference(model)

subjects = ["Biology", "Chemistry", "Physics", "History", "Computer Science",
            "Mathematics", "Psychology", "Economics", "Literature"]

pairs = []
attempts = 0
TARGET_PAIRS = 2000
MAX_ATTEMPTS = TARGET_PAIRS * 3

print(f"Generating {TARGET_PAIRS} preference pairs...")
while len(pairs) < TARGET_PAIRS and attempts < MAX_ATTEMPTS:
    attempts += 1
    subject = random.choice(subjects)
    difficulty = random.choice(["easy", "medium", "hard", "mixed"])
    grade = random.choice(["Grade 8", "Grade 10", "Undergraduate"])
    num_q = random.choice([3, 5])

    prompt = f"Generate {num_q} {difficulty} MCQ questions about {subject} for {grade} students. Output valid JSON."
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    responses, scores = [], []
    for _ in range(5):
        output = model.generate(**inputs, max_new_tokens=1024, temperature=0.8, top_p=0.9, do_sample=True)
        text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        responses.append(text)
        scores.append(_score_quiz(text))

    if not scores:
        continue
    best_idx, worst_idx = scores.index(max(scores)), scores.index(min(scores))
    if scores[best_idx] - scores[worst_idx] > 0.2:
        pairs.append({"prompt": prompt, "chosen": responses[best_idx], "rejected": responses[worst_idx]})

    if len(pairs) % 100 == 0 and len(pairs) > 0:
        print(f"  {len(pairs)}/{TARGET_PAIRS} pairs ({attempts} attempts)")

FastLanguageModel.for_training(model)
preference_dataset = Dataset.from_list(pairs)
print(f"Generated {len(pairs)} preference pairs from {attempts} attempts")

In [ ]:
simpo_config = SimPOConfig(
    output_dir="./stage3_simpo",
    learning_rate=5e-7,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    beta=2.0,
    gamma=1.0,
    bf16=True,
    max_length=2048,
    max_prompt_length=512,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to="none",
)

trainer = SimPOTrainer(
    model=model, args=simpo_config,
    train_dataset=preference_dataset,
    tokenizer=tokenizer,
)

print("Starting Stage 3 SimPO...")
stats = trainer.train()
print(f"Done! Steps: {stats.global_step}")

model.save_pretrained("./stage3_simpo/final")
tokenizer.save_pretrained("./stage3_simpo/final")
print("Stage 3 saved.")

## 5. Evaluation

Compare Base vs Stage 1 vs Stage 2 vs Stage 3.

In [ ]:
eval_models = {
    "base": "unsloth/gemma-4-E4B-it",
    "stage1_sft": "./stage1_sft/final",
    "stage2_grpo": "./stage2_grpo/final",
    "stage3_simpo": "./stage3_simpo/final",
}

NUM_EVAL = 100  # Keep manageable on T4
eval_subset = eval_data[:NUM_EVAL]
all_results = []

for name, path in eval_models.items():
    if not Path(path).exists() and not path.startswith("unsloth/"):
        print(f"Skipping {name}: not found")
        continue
    print(f"\nEvaluating: {name}")
    m, tok = FastLanguageModel.from_pretrained(path, max_seq_length=2048, load_in_4bit=True)
    FastLanguageModel.for_inference(m)

    metrics = {"json_valid": 0, "schema_valid": 0, "has_explanation": 0,
               "has_bloom": 0, "avg_reward": 0.0, "total": 0}

    for ex in eval_subset:
        prompt = ex["messages"][0]["content"]
        inputs = tok(prompt, return_tensors="pt").to(m.device)
        output = m.generate(**inputs, max_new_tokens=1024, temperature=0.3, do_sample=True)
        response = tok.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        metrics["total"] += 1

        parsed = _try_parse_json(response)
        if parsed is not None:
            metrics["json_valid"] += 1
            questions = parsed.get("questions", [])
            if isinstance(questions, list) and len(questions) > 0:
                metrics["schema_valid"] += 1
                if any(q.get("explanation") and len(q["explanation"]) > 20 for q in questions if isinstance(q, dict)):
                    metrics["has_explanation"] += 1
                if any(q.get("bloom_level") for q in questions if isinstance(q, dict)):
                    metrics["has_bloom"] += 1
        metrics["avg_reward"] += _score_quiz(response)

    t = metrics["total"]
    if t > 0:
        metrics["avg_reward"] /= t
    result = {
        "stage": name,
        "json%": f"{metrics['json_valid']/t*100:.1f}%",
        "schema%": f"{metrics['schema_valid']/t*100:.1f}%",
        "explain%": f"{metrics['has_explanation']/t*100:.1f}%",
        "bloom%": f"{metrics['has_bloom']/t*100:.1f}%",
        "reward": f"{metrics['avg_reward']:.3f}",
    }
    all_results.append(result)
    print(f"  {result}")

print("\n" + "="*70)
print("EVALUATION RESULTS — SmartStudy AI")
print("="*70)
print(f"{'Stage':<15} {'JSON%':<10} {'Schema%':<10} {'Explain%':<10} {'Bloom%':<10} {'Reward':<10}")
print("-"*65)
for r in all_results:
    print(f"{r['stage']:<15} {r['json%']:<10} {r['schema%']:<10} {r['explain%']:<10} {r['bloom%']:<10} {r['reward']:<10}")

## 6. Export to GGUF for Ollama

Export the final Stage 3 model as GGUF (q4_k_m) for local deployment on M3 Pro.

In [ ]:
print("Loading final model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    "./stage3_simpo/final", max_seq_length=2048, load_in_4bit=True,
)

EXPORT_NAME = "smartstudy-gemma4-edu"
QUANT = "q4_k_m"

print(f"Exporting to GGUF ({QUANT})...")
model.save_pretrained_gguf(EXPORT_NAME, tokenizer, quantization_method=QUANT)
print(f"\nGGUF export complete: {EXPORT_NAME}/")
print(f"\nDeploy locally:")
print(f"  1. Download the GGUF file from Kaggle output")
print(f"  2. Create a Modelfile: FROM ./{EXPORT_NAME}-unsloth.{QUANT.upper()}.gguf")
print(f"  3. ollama create smartstudy-edu -f Modelfile")
print(f"  4. ollama run smartstudy-edu")

In [ ]:
# Optional: Push to HuggingFace Hub
# Uncomment and set your token for Unsloth $10K prize eligibility

# HF_TOKEN = "hf_..."
# model.push_to_hub_gguf(
#     "YOUR_USERNAME/smartstudy-gemma4-edu",
#     tokenizer,
#     quantization_method="q4_k_m",
#     token=HF_TOKEN,
# )
# print("Pushed to HuggingFace Hub!")

## Summary

| Stage | What it does | Method |
|-------|-------------|--------|
| **Stage 1** | Teach format + domain | SFT + rsLoRA (r=32) |
| **Stage 2** | Teach quality reasoning | GRPO (8 generations, reward functions) |
| **Stage 3** | Teach taste/preference | SimPO (self-play, reference-free) |
| **Export** | Deploy locally | GGUF q4_k_m for Ollama |

The GGUF file can be downloaded from Kaggle output and deployed with `ollama create`.